In [0]:
import requests
import base64
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import json
import time
from pyspark.sql.functions import to_date, col
from pyspark.sql.functions import current_timestamp
import logging
import concurrent.futures
import os
import re
from urllib.parse import urlparse

In [0]:
def get_logger(name=default_logger_name):
    logger = logging.getLogger(name)
    if not logger.hasHandlers():
        logger.setLevel(logging.INFO)
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            '%(asctime)s %(levelname)s %(name)s: %(message)s'
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger
    
logger = get_logger()

In [0]:

def valid_input_check(**kwargs):
    if "url" in kwargs:
        valid_url_check(kwargs["url"])
    if "volume_path" in kwargs:
        valid_volume_path_check(kwargs["volume_path"])
    if "request_mode" in kwargs:
        valid_request_mode_check(kwargs["request_mode"])
    if "file_name" in kwargs:
        valid_name_check(kwargs["file_name"])
    if "start_dt" in kwargs and "end_dt" in kwargs:
        valid_date_range_check(kwargs["start_dt"], kwargs["end_dt"])
    return True

def valid_url_check(url: str):
    if not url or url.strip() == "":
        raise ValueError("URL cannot be empty.")
    parsed = urlparse(url)
    if parsed.scheme.lower() != "https":
        raise ValueError("URL must start with https://")
    pattern = r"^https:\/\/[A-Za-z0-9\.\-\/\?\=\&\%]+$"
    if not re.match(pattern, url):
        raise ValueError("Invalid URL format.")
    return True

def valid_volume_path_check(path: str):
    if not path or path.strip() == "":
        raise ValueError("Volume path cannot be empty.")
    if " " in path:
        raise ValueError("Volume path cannot contain spaces.")
    if not path.startswith("/Volumes/"):
        raise ValueError("Invalid volume path format. Must start with /Volumes/")
    return True

def valid_request_mode_check(mode: str):
    if not mode:
        raise ValueError("Request mode cannot be empty.")
    if not mode.isalpha():
        raise ValueError("Request mode must contain only letters.")
    if mode.upper() not in allowed_modes:
        raise ValueError(f"Invalid request mode. Allowed: {allowed_modes}")
    return True

def valid_name_check(name: str):
    if not name or name.strip() == "":
        raise ValueError("Name cannot be empty.")
    if name != name.strip():
        raise ValueError("Name cannot contain leading or trailing spaces.")
    return True

def valid_date_range_check(start_dt: str, end_dt: str):
    fmt = "%Y-%m-%d"
    try:
        s = datetime.strptime(start_dt, fmt).date()
        e = datetime.strptime(end_dt, fmt).date()
    except:
        raise ValueError("Dates must be in YYYY-MM-DD format.")
    if s > e:
        raise ValueError("Start date cannot be after end date.")
    if e > date.today():
        raise ValueError("End date cannot be in the future.")
    return True

In [0]:
current_token = None
token_expiry = 0

def token_header_content(auth_b64, content_type):
    # Creates headers for token generation requests.
    headers = {
        "Authorization": f"Basic {auth_b64}",
        "Content-Type": content_type
    }
    return headers

def create_headers_token(client_id, client_secret):
    """Creates authorization headers for token generation.
    Args:
        client_id (str): The client ID for API authentication
        client_secret (str): The client secret for API authentication
    Returns:
        dict: Headers dictionary containing Authorization and Content-Type
    Raises: Exception: If encoding fails or parameters are invalid"""
    try:
        auth_str = f"{client_id}:{client_secret}"
        auth_bytes = auth_str.encode("utf-8")
        auth_b64 = base64.b64encode(auth_bytes).decode("utf-8")
        headers = token_header_content(auth_b64, content_type)
        return headers
    except Exception as e:
        raise Exception(f"Error in create_headers_token: {str(e)}")

def create_data_token(access_key, access_secret):
    """Creates data payload for token generation request.
    Args:
        access_key (str): The access key username
        access_secret (str): The access secret password
    Returns:
        dict: Data dictionary with grant_type, username, password, and scope
    Raises:
        Exception: If data creation fails."""
    try:
        data = {
            "grant_type": "password",
            "username": access_key,
            "password": access_secret,
            "scope": scope
        }
        return data
    except Exception as e:
        raise Exception(f"Error in create_data_token: {str(e)}")


def get_token():
    """Retrieves an access token from the Inovalon API with intelligent caching.
    This function implements token caching to avoid unnecessary token generation requests.
    It reuses existing tokens if they are still valid and only fetches new tokens when:
    - No token exists
    - Current token is expiring within 5 minutes
    Returns:
        str: The access token string
    Raises:
        Exception: If token retrieval fails or returns non-200 status"""
    global current_token, token_expiry
    try:
        now = time.time()
        # If token exists & not expiring in next 5 minutes → reuse it
        if current_token and now < token_expiry - 300:
            remaining = int((token_expiry - now) / 60)
            return current_token

        headers = create_headers_token(client_id, client_secret)
        data = create_data_token(access_key, access_secret)
        response = requests.post(token_url, headers=headers, data=data, timeout=60)
        
        if response.status_code != 200:
            error_msg = f"Token generation FAILED: {response.status_code} {response.text}"
            raise Exception(error_msg)
        json_resp = response.json()
        current_token = json_resp["access_token"]

        # Get expiry time (default to 3600 seconds if not provided)
        expires_in = json_resp.get("expires_in", 3600)
        token_expiry = now + expires_in

        expiry_local = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(token_expiry))
        return current_token
        
    except requests.exceptions.Timeout:
        raise Exception("Token request timed out after 60 seconds")
    except requests.exceptions.RequestException as e:
        raise Exception(f"Network error in get_token: {str(e)}")
    except KeyError as e:
        raise Exception(f"Invalid token response format: {str(e)}")
    except Exception as e:
        raise Exception(f"Error in get_token: {str(e)}")

In [0]:
def create_request_headers(request_mode):
    """Creates headers for API requests to Inovalon endpoints.
    Args:
        request_mode (str): The request mode for the API call
        account_key (str): The account key for authentication
        token (str): The bearer token for authorization
    Returns:
        dict: Headers dictionary containing accept, X-AccountKey, Authorization, and X-Request-Mode
    Raises: Exception: If header creation fails."""
    try:
        headers = {
            "accept": accept_key,
            "X-AccountKey": f"{account_key}",
            "Authorization": f"Bearer {get_token()}",
            "X-Request-Mode": request_mode
        }
        return headers
        
    except Exception as e:
        raise Exception(f"Error in create_request_headers: {str(e)}")

def date_check(start_dt, end_dt):
    # Validates and parses start and end date strings, ensuring start <= end.
    try:
        if start_dt > end_dt:
            logger.error("startDate should be lesser than endDate")
            raise ValueError("startDate should be lesser than endDate")
    except Exception as ex:
        logger.error(f"Error in date_check: {str(ex)}")
        raise ex

def make_get_request(url, headers=None, params=None, retries=3):
    """ Makes a GET request to the specified URL with error handling, 3 retries.
    Args:
        url (str): The URL to send the GET request to
        headers (dict, optional): Headers to include in the request
        params (dict, optional): Query parameters for the request
    Returns:
        dict: JSON response from the API, or None if request fails
    Raises: None: Returns None on failure and logs the error."""
    try:
        for attempt in range(retries):
            try:
                logger.info(f"GET Attempt {attempt}/{retries}: {url}")
                response = requests.get(url, headers=headers, params=params, timeout=120)
                response.raise_for_status()
                return response.json()
            except (requests.exceptions.Timeout, requests.exceptions.HTTPError, requests.exceptions.RequestException, Exception) as e:
                logger.error(f"Attempt {attempt} failed: {str(e)}")
                if attempt < retries - 1:
                    sleep_time = 5
                    time.sleep(sleep_time)
                else:
                    raise
    except Exception as e:
        logger.error(f"GET request failed after {retries} attempts: {url}")
        raise Exception(f"GET request failed after {retries} attempts: {url}")


def generate_chunks(start_dt, end_dt, mode):
    """ Generates date chunks based on the specified mode for batch processing.
    Args:
        start_dt (datetime): The start date for chunking
        end_dt (datetime): The end date for chunking
        mode (str): The chunking mode - "daily", "monthly", "quarterly", or "yearly"
    Returns:
        list: List of tuples containing (start_date_str, end_date_str) for each chunk
    Raises:
        ValueError: If an invalid mode is provided
        Exception: If date chunking fails."""
    try:
        chunks = []
        start_timestamp = start_dt
        while start_timestamp <= end_dt:
            if mode == chunk_type[0]:
                end_timestamp = start_timestamp
                next_start = start_timestamp + timedelta(days=1)
            elif mode == chunk_type[1]:
                end_timestamp = (start_timestamp + relativedelta(months=1)) - timedelta(days=1)
                next_start = start_timestamp + relativedelta(months=1)
            elif mode == chunk_type[2]:
                end_timestamp = (start_timestamp + relativedelta(months=3)) - timedelta(days=1)
                next_start = start_timestamp + relativedelta(months=3)
            elif mode == chunk_type[3]:
                end_timestamp = (start_timestamp + relativedelta(years=1)) - timedelta(days=1)
                next_start = start_timestamp + relativedelta(years=1)
            else:
                logger.error(f"Invalid mode '{mode}'. Choose from " + ", ".join(chunk_type))
                raise ValueError(f"Invalid mode '{mode}'. Choose from " + ", ".join(chunk_type))
            if end_timestamp > end_dt:
                end_timestamp = end_dt
            chunks.append((
                start_timestamp.strftime("%Y-%m-%d"),
                end_timestamp.strftime("%Y-%m-%d")
            ))
            start_timestamp = next_start
        logger.info(f"Total chunks generated: {len(chunks)}")
        return chunks
    except ValueError as ve:
        logger.error(f"Error in generate_chunks: {str(ve)}")
        raise ValueError(f"Error in generate_chunks: {str(ve)}")
    except Exception as e:
        logger.error(f"Error in generate_chunks: {str(e)}")
        raise Exception(f"Error in generate_chunks: {str(e)}")

def generate_url(url_type, **kwargs):
    try:
        if url_type == url_type_config[0]:
            url = (f"{kwargs['endpoint_url']}?created={kwargs['start']},{kwargs['end']}"
                   f"&limit={kwargs['limit']}&offset={kwargs['offset']}")
            return url
        elif url_type == url_type_config[1]:
            url = f"{kwargs['url']}/{kwargs['id']}"
            return url
        else:
            logger.error(f"Invalid url_type: {url_type}")
            raise ValueError("Invalid url_type")
    except Exception as e:
        logger.error(f"Error in generate_url: {str(e)}")
        raise Exception(f"Error in generate_url: {str(e)}")

def fetch_records(endpoint_url, headers, start_timestamp, end_timestamp, limit=200, max_offset=10000):
    """ Fetches all records from an API endpoint for a given date range with pagination.
    Args:
        endpoint_url (str): The API endpoint URL
        headers (dict): Headers for the API request
        start_timestamp (str): Start date in YYYY-MM-DD format
        end_timestamp (str): End date in YYYY-MM-DD format
        limit (int, optional): Number of records per page. Defaults to 200
        max_offset (int, optional): Maximum offset for pagination. Defaults to 10000
    Returns:
        list: List of all fetched records
    Raises: Exception: If fetching records fails"""
    try:
        all_records = []
        offset = 0
        while True:
            url = generate_url(url_type=url_type_config[0], endpoint_url=endpoint_url, start=start_timestamp, end=end_timestamp, limit=limit, offset=offset)
            logger.info(f"Fetching records with offset {offset}: {url}")
            data = make_get_request(url, headers)
            records = data.get("records", []) if data else []
            logger.info(f"Fetched {len(records)} records at offset {offset}")
            if not records:
                logger.info(f"No more records found at offset {offset}. Stopping pagination.")
                break   
            all_records.extend(records)
            offset += limit
            if offset >= max_offset or len(records) < limit:
                logger.info(f"Reached max_offset or last page at offset {offset}. Stopping pagination.")
                break
        logger.info(f"Total records fetched: {len(all_records)}")
        return all_records
    except Exception as e:
        logger.error(f"Error in fetch_records: {str(e)}")
        raise Exception(f"Error in fetch_records: {str(e)}")

def write_json_to_dbfs(full_path, json_str):
    """ Writes a JSON string to the specified DBFS path."""
    try:
        with open(full_path, "w") as f:
            f.write(json_str)
        logger.info(f"Successfully wrote JSON to DBFS: {full_path}")
        return True
    except Exception as e:
        logger.error(f"Error writing JSON to DBFS: {str(e)}")
        raise Exception(f"Error writing JSON to DBFS: {str(e)}")

def save_list_records_to_dbfs(records, file_name, start_timestamp, end_timestamp, base_path):
    """Saves API records to DBFS as a JSON file for the given date range."""
    try:
        json_str = json.dumps(records, indent=2)
        file_name = f"{file_name}_{start_timestamp}_{end_timestamp}.json"
        full_path = base_path + file_name
        write_json_to_dbfs(full_path, json_str)
        return True
    except Exception as e:
        logger.error(f"Error saving records to DBFS: {str(e)}")
        raise Exception(f"Error saving records to DBFS: {str(e)}")

def request_endpoint(start_date_str,
                    end_date_str,
                    url,
                    chunks,
                    request_mode,
                    file_name,
                    volume_path):
    """Requests data from API endpoint for all date chunks and saves to DBFS volume.
    Args:
        start_date_str (str): Start date string
        end_date_str (str): End date string
        url (str): API endpoint URL
        chunks (list): List of date range tuples
        request_mode (str): Request mode for API
        name (str): Name prefix for saved files
        volume_path (str): Path to DBFS volume for saving files
        account_key (str): Account key for authentication
        token (str): Bearer token for authorization
    Returns:
        str: Success or failure message
    Raises: Exception: If saving data fails."""
    try:
        api_url = url
        base_path = volume_path
        saved_files = 0
        failed_chunks = []
        for start_timestamp, end_timestamp in chunks:
            try:
                headers = create_request_headers(request_mode)
                records = fetch_records(api_url, headers, start_timestamp, end_timestamp)
                if records:                    
                    status = save_list_records_to_dbfs(records, file_name, start_timestamp, end_timestamp, base_path)
                    if status:
                        saved_files += 1

            except Exception as e:
                logger.error(f"Failed to process chunk {start_timestamp} to {end_timestamp}: {str(e)}")
                failed_chunks.append((start_timestamp, end_timestamp))
                continue
        if failed_chunks:
            logger.error(f"Partially saved to DBFS volume. {saved_files} files saved, {len(failed_chunks)} chunks failed.")
        logger.info(f"Saved to DBFS volume successfully. Total files: {saved_files}")
    except Exception as e:
        logger.error(f"Failed to save data to DBFS: {str(e)}")
        raise Exception(f"Failed to save data to DBFS: {str(e)}")

    
def list_extraction(url,start_dt,end_dt,mode,request_mode,file_name,volume_path):
    """Main function to extract data from Inovalon API and save to DBFS volume.
    This function orchestrates the entire extraction process:
    1. Validates and parses date parameters
    2. Generates date chunks based on mode
    3. Retrieves authentication token
    4. Fetches data for each chunk
    5. Saves data to DBFS volume
    Returns:
        str: Success or failure message with details
    Raises:
        Exception: If extraction process fails at any stage."""

    try:
        start_dt = datetime.strptime(start_dt, "%Y-%m-%d")
        end_dt = datetime.strptime(end_dt, "%Y-%m-%d")
        date_check(start_dt, end_dt)
        chunks = generate_chunks(start_dt, end_dt, mode)
        result= request_endpoint(start_dt, end_dt,url, chunks, request_mode, file_name, volume_path)
        return result
    except ValueError as ve:
        raise Exception(f"Invalid date parameters: {str(ve)}")
    except Exception as e:
        raise Exception(f"Extraction failed: {str(e)}")

In [0]:
def id_extraction(url,file_name,valid_json_path,corrupt_json_path,id_table,list_table,limit,request_mode):
    id_list=get_ids(list_table,id_table,limit)
    if id_list:
        url_list = [ generate_url(url_type_config[1], url=url, id=id) for id in id_list ]
    else:
        url_list=[]
    headers = create_request_headers(request_mode)
    max_threads = min(32, (os.cpu_count() or 1) * 2)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_threads) as executor:
        results = list(executor.map(lambda u: fetch_url(u, headers), url_list))
        
    dbfs_load(results,file_name,valid_json_path,corrupt_json_path)

def get_ids(list_table,id_table,limit):
    unique_id_qry = f"""
        SELECT a.id
        FROM (
            SELECT *
            FROM {list_table}
            WHERE `_active_flag` = 1
        ) a
        LEFT OUTER JOIN (
            SELECT *
            FROM {id_table}
            WHERE `_active_flag` = 1
        ) b
        ON a.id = b.TransID
        WHERE b.TransId IS NULL
        ORDER BY create_date DESC
        LIMIT {limit}
    """
    df = spark.sql(unique_id_qry)
    id_list = [row.id for row in df.collect() if row.id is not None]
    return id_list

def fetch_url(u,headers):
    return make_get_request(u, headers=headers, params=None, retries=3)

def dbfs_load(results,file_name,valid_json_path,corrupt_json_path):
    valid_results = []
    corrupt_results = []

    start_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    valid_file_name = f"{file_name}_{start_timestamp}.json"
    corr_file_name = f"err_{file_name}_{start_timestamp}.json"
    
    for res in results:
        try:
            print(type(res))
            if isinstance(res, dict):
                valid_results.append(json.dumps(res))
            else:
                corrupt_results.append(res)
        except Exception:
            corrupt_results.append(res)
    
    if valid_results:
        dbutils.fs.put(valid_json_path + valid_file_name, "\n".join(valid_results), overwrite=True)
    if corrupt_results:
        dbutils.fs.put(corrupt_json_path + corr_file_name, "\n".join(corrupt_results), overwrite=True)